# Getting Started: Load and Analyze Your First Signal

This notebook demonstrates the core workflow of `predictive-maintenance-mcp`:
1. Generate a synthetic vibration signal
2. Load it into the Signal Repository
3. Run spectral analysis (FFT, PSD, STFT)
4. Assess vibration severity per ISO 10816

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys, os

# Add project root to path
project_root = Path('.').resolve().parent
sys.path.insert(0, str(project_root / 'src'))
os.chdir(project_root)

print(f'Project root: {project_root}')

## Step 1: Generate a Test Signal

We create a synthetic bearing vibration signal with known characteristics.

In [ ]:
fs = 10000  # 10 kHz sampling rate
duration = 2.0  # 2 seconds
t = np.arange(0, duration, 1/fs)

# Shaft rotation: 30 Hz (1800 RPM)
f_shaft = 30.0

# Simulate bearing outer race fault at BPFO = 120 Hz
bpfo = 120.0
signal = (
    0.5 * np.sin(2 * np.pi * f_shaft * t)           # shaft imbalance
    + 0.3 * np.sin(2 * np.pi * bpfo * t)            # BPFO fundamental
    + 0.15 * np.sin(2 * np.pi * 2 * bpfo * t)       # 2x BPFO harmonic
    + 0.08 * np.sin(2 * np.pi * 3 * bpfo * t)       # 3x BPFO harmonic
    + 0.2 * np.random.randn(len(t))                   # noise
)

print(f'Signal: {len(signal)} samples, {duration}s at {fs} Hz')
print(f'Expected peaks: shaft={f_shaft} Hz, BPFO={bpfo} Hz, 2xBPFO={2*bpfo} Hz')

In [ ]:
# Save to CSV
signal_dir = project_root / 'data' / 'signals'
signal_dir.mkdir(parents=True, exist_ok=True)
signal_path = signal_dir / 'tutorial_bearing.csv'
pd.DataFrame(signal).to_csv(signal_path, header=False, index=False)

# Save metadata
import json
meta_path = signal_dir / 'tutorial_bearing_metadata.json'
with open(meta_path, 'w') as f:
    json.dump({'sampling_rate': fs, 'signal_unit': 'g'}, f)

print(f'Saved: {signal_path}')

## Step 2: Load into Signal Repository

The Signal Repository caches signals in memory for fast repeated analysis.

In [ ]:
from predictive_maintenance_mcp.signal_acquisition import SignalRepository

repo = SignalRepository(max_memory_bytes=1 * 1024**3)  # 1 GB cap
info = repo.load_signal(str(signal_path), signal_id='tutorial_bearing', sampling_rate=fs)

print(f'Loaded signal: {info["signal_id"]}')
print(f'  Samples: {info["num_samples"]}')
print(f'  Duration: {info["duration_s"]}s')
print(f'  Size: {info["size_bytes"] / 1024:.1f} KB')

## Step 3: Spectral Analysis

In [ ]:
from predictive_maintenance_mcp.signal_processing import compute_psd, compute_stft_spectrogram

# Power Spectral Density (Welch)
psd = compute_psd(signal, fs, nperseg=1024, num_peaks=10)

print('Top PSD peaks:')
for i, peak in enumerate(psd['top_peaks'][:5], 1):
    print(f'  {i}. {peak["frequency_hz"]:8.2f} Hz  ({peak["magnitude_db"]:+.1f} dB)')

print(f'\nTotal power: {psd["total_power"]:.4f}')
print(f'Frequency resolution: {psd["frequency_resolution"]:.2f} Hz')

In [ ]:
# STFT Spectrogram summary
stft_result = compute_stft_spectrogram(signal, fs, nperseg=512)

print('STFT Spectrogram summary:')
print(f'  Time bins: {stft_result["num_time_bins"]}')
print(f'  Freq bins: {stft_result["num_freq_bins"]}')
print(f'  Max power at: {stft_result["max_power_freq_hz"]:.1f} Hz, t={stft_result["max_power_time_s"]:.3f}s')
print('\nEnergy per band:')
for band in stft_result['energy_per_band']:
    print(f'  {band["band"]:>15s}: {band["energy"]:.4f}')

## Step 4: ISO 10816 Severity Assessment

In [ ]:
from predictive_maintenance_mcp.diagnostics import assess_vibration_severity

severity = assess_vibration_severity(signal, fs, machine_class='II', signal_unit='g')

print(f'ISO 10816 Assessment:')
print(f'  Zone: {severity["zone"]} ({severity["severity"]})')
print(f'  Velocity RMS: {severity["velocity_rms_mm_s"]:.2f} mm/s')
print(f'  Recommendation: {severity["recommendation"]}')

## Summary

In this notebook we:
1. Generated a synthetic bearing fault signal with known frequencies
2. Loaded it into the Signal Repository for efficient caching
3. Ran PSD and STFT analysis to identify dominant frequencies
4. Assessed vibration severity per ISO 10816

**Next**: See `02_bearing_diagnostics.ipynb` for bearing-specific fault detection.